<a id="huygens-fresnel"></a>

$$
\text{衍射 (Diffraction): 光绕过障碍物 / 穿过小孔后的分布} \\ \quad \\
\text{惠更斯-菲涅尔原理 (Huygens-Fresnel Principle): 孔径上的每一点都是一个新的子波源, 屏幕上的光场是所有子波的叠加 (积分)} \\
\text{夫琅禾费衍射 (Fraunhofer Diffraction): 屏幕很远 (或在透镜焦平面上), 到达同一个方向 } \theta \text{ 的光线近似平行} \\
\text{孔径上位置 } x' \text{ 的子波, 相对于 } x' = 0 \text{ 的子波, 光程少了 } x'\sin\theta \text{ (和双缝的推导一样)} \\
E(\theta) \propto \int A(x') \, e^{-i k x' \sin\theta} \, dx', \quad k = \frac{2\pi}{\lambda} \\
A(x') \text{: 孔径函数 (Aperture Function), 透光处为 1, 不透光处为 0} \\ \quad \\
\text{记 } k_x = k\sin\theta \text{ (或空间频率 } f_x = \frac{\sin\theta}{\lambda}\text{), 这正是孔径函数的傅里叶变换 (Fourier Transform):} \\
E(f_x) \propto \hat{A}(f_x) = \int A(x') e^{-2\pi i f_x x'} dx' \\
\text{夫琅禾费衍射图样 = 孔径的傅里叶变换的模平方, 孔越小, 图样越宽 (傅里叶变换的尺度关系)}
$$

数学基础: [欧拉公式](../math_basics/calculus.ipynb#taylor) · [定积分](../math_basics/calculus.ipynb#integral) · 应用: [杨氏双缝](interference.ipynb#young)

<a id="single-slit"></a>

$$
\text{单缝 (Single Slit), 宽度 a: } A(x') = 1 \text{ 当 } |x'| \leq \frac{a}{2} \\
E(\theta) \propto \int_{-a/2}^{a/2} e^{-i k x'\sin\theta} dx' = \left[\frac{e^{-ik x'\sin\theta}}{-ik\sin\theta}\right]_{-a/2}^{a/2} = \frac{e^{ika\sin\theta/2} - e^{-ika\sin\theta/2}}{ik\sin\theta} \\
\text{欧拉公式 } e^{i\phi} - e^{-i\phi} = 2i\sin\phi \implies E(\theta) \propto \frac{2\sin\left(\frac{ka\sin\theta}{2}\right)}{k\sin\theta} = a\frac{\sin\beta}{\beta}, \quad \beta = \frac{ka\sin\theta}{2} = \frac{\pi a\sin\theta}{\lambda} \\
I(\theta) = I_0\left(\frac{\sin\beta}{\beta}\right)^2 = I_0\,\text{sinc}^2\beta \quad (\theta = 0 \text{ 时 } \frac{\sin\beta}{\beta} \to 1) \\ \quad \\
\text{暗纹 (Minima): } \sin\beta = 0, \ \beta \neq 0 \implies \beta = m\pi \implies a\sin\theta = m\lambda, \ m = \pm 1, \pm 2, \cdots \\
\text{直观: 把缝分成上下两半, 每一对相距 } \frac{a}{2} \text{ 的点的光程差都是 } \frac{\lambda}{2} \text{, 两两抵消} \\
\text{中央亮纹的角宽度 } 2\arcsin\frac{\lambda}{a} \text{, 缝越窄越宽}; \text{ 次级亮纹约在 } \tan\beta = \beta \text{ 处, 强度只有中央的 4.7\%}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import brentq
from scipy.special import j1

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

wavelength = 500e-9
k = 2 * np.pi / wavelength
a = 5e-6

def single_slit_numeric(theta):
    # 直接数值积分惠更斯-菲涅尔积分 (实部和虚部分别积分)
    s = k * np.sin(theta)
    re, _ = quad(lambda x: np.cos(s * x), -a / 2, a / 2)
    im, _ = quad(lambda x: -np.sin(s * x), -a / 2, a / 2)
    return re + 1j * im

theta = np.linspace(-0.4, 0.4, 801)
E_num = np.array([single_slit_numeric(t) for t in theta])
beta = np.pi * a * np.sin(theta) / wavelength
E_formula = a * np.sinc(beta / np.pi)            # np.sinc(x) = sin(pi x) / (pi x)
print(f'数值积分 == a sin(beta)/beta ? {np.allclose(E_num, E_formula, atol=1e-12 * a)}')

for m in [1, 2, 3]:
    th_m = np.arcsin(m * wavelength / a)
    print(f'm = {m}: 暗纹 theta = {np.degrees(th_m):.4f}°, |E|^2 = {abs(single_slit_numeric(th_m))**2 / a**2:.2e}')

# 第一个次级亮纹: tan(beta) = beta
beta1 = brentq(lambda b: np.tan(b) - b, np.pi + 0.1, 1.5 * np.pi - 0.01)
print(f'第一个次级亮纹 beta = {beta1:.4f} (约 1.43 pi), 相对强度 = {(np.sin(beta1) / beta1)**2:.4f}')

plt.figure(figsize=(4, 3))
plt.plot(np.degrees(theta), np.abs(E_formula)**2 / a**2, label='sinc^2 formula')
plt.plot(np.degrees(theta[::20]), np.abs(E_num[::20])**2 / a**2, 'o', markersize=3, label='numerical integral')
for m in [-2, -1, 1, 2]:
    plt.axvline(np.degrees(np.arcsin(m * wavelength / a)), color='gray', linestyle=':')
plt.xlabel('theta (deg)')
plt.ylabel('I / I0')
plt.title('single slit')
plot_nicely()
plt.show()

<a id="grating"></a>

$$
\text{N 缝 / 衍射光栅 (Diffraction Grating): N 条宽度 a 的缝, 间距 (周期) d} \\
\text{第 n 条缝的位置平移了 } nd \text{, 傅里叶变换的平移性质: 每条缝的贡献多乘一个相位 } e^{-iknd\sin\theta} \\
E(\theta) = E_{\text{单缝}}(\theta) \sum_{n=0}^{N-1} e^{-in\phi}, \quad \phi = kd\sin\theta \\
\text{等比数列求和: } \sum_{n=0}^{N-1} q^n = \frac{1 - q^N}{1 - q}, \ q = e^{-i\phi} \\
\left|\frac{1 - e^{-iN\phi}}{1 - e^{-i\phi}}\right| = \left|\frac{e^{-iN\phi/2}(e^{iN\phi/2} - e^{-iN\phi/2})}{e^{-i\phi/2}(e^{i\phi/2} - e^{-i\phi/2})}\right| = \left|\frac{\sin(N\phi/2)}{\sin(\phi/2)}\right| \\
I(\theta) = I_0\underbrace{\left(\frac{\sin\beta}{\beta}\right)^2}_{\text{单缝衍射 (包络)}}\underbrace{\left(\frac{\sin N\alpha}{N\sin\alpha}\right)^2}_{\text{多缝干涉}}, \quad \alpha = \frac{\phi}{2} = \frac{\pi d\sin\theta}{\lambda} \\ \quad \\
\text{主极大 (Principal Maxima): } \sin\alpha = 0 \text{, 用洛必达法则 } \frac{\sin N\alpha}{N\sin\alpha} \to \pm 1 \\
\alpha = m\pi \implies d\sin\theta = m\lambda \quad (\text{光栅方程 Grating Equation}) \\
\text{两个主极大之间有 N - 1 个零点 } (N\alpha = \text{整数} \times \pi) \text{, 所以 N 越大, 亮线越细, 越亮 (峰值} \propto N^2\text{)} \\
\text{N = 2 时 } \left(\frac{\sin 2\alpha}{2\sin\alpha}\right)^2 = \cos^2\alpha \text{, 就是双缝干涉}
$$

In [ ]:
d_grating = 20e-6

def grating_intensity(theta, N):
    beta = np.pi * a * np.sin(theta) / wavelength
    alpha = np.pi * d_grating * np.sin(theta) / wavelength
    single = np.sinc(beta / np.pi)**2
    # sin(N alpha) / (N sin alpha), 在 sin(alpha) = 0 处取极限 +-1
    with np.errstate(invalid='ignore', divide='ignore'):
        multi = np.where(np.abs(np.sin(alpha)) < 1e-12, 1.0, (np.sin(N * alpha) / (N * np.sin(alpha)))**2)
    return single * multi

def grating_direct_sum(theta, N):
    # 直接把 N 条缝的单缝场加起来 (每条缝平移 n d), 不用等比数列公式
    E_single = a * np.sinc(np.pi * a * np.sin(theta) / wavelength / np.pi)
    phases = np.exp(-1j * k * np.outer(np.sin(theta), np.arange(N) * d_grating))
    E = E_single * phases.sum(axis=1)
    return np.abs(E)**2 / (N * a)**2

theta = np.linspace(-0.12, 0.12, 20001)
for N in [2, 5, 20]:
    print(f'N = {N:>2}: 等比数列公式 == 直接求和? {np.allclose(grating_intensity(theta, N), grating_direct_sum(theta, N), atol=1e-12)}')

alpha2 = np.pi * d_grating * np.sin(theta) / wavelength
print(f'N = 2 时多缝因子 == cos^2(alpha) ? {np.allclose(grating_direct_sum(theta, 2) / np.sinc(np.pi * a * np.sin(theta) / wavelength / np.pi)**2, np.cos(alpha2)**2)}')

# 主极大位置 d sin(theta) = m lambda
I20 = grating_intensity(theta, 20)
peak_idx = np.where((I20[1:-1] > I20[:-2]) & (I20[1:-1] > I20[2:]) & (I20[1:-1] > 0.05))[0] + 1
print(f'N = 20 主极大的 d sin(theta) / lambda = {np.round(d_grating * np.sin(theta[peak_idx]) / wavelength, 3)}')
print(f'd / a = {d_grating / a:.0f}: 第 4 级主极大正好落在单缝暗纹上 (缺级 missing order)')

plt.figure(figsize=(8, 3))
for N in [2, 5, 20]:
    plt.plot(np.degrees(theta), grating_intensity(theta, N), label=f'N = {N}', linewidth=0.8)
plt.plot(np.degrees(theta), np.sinc(np.pi * a * np.sin(theta) / wavelength / np.pi)**2, 'k--', label='single slit envelope')
plt.xlabel('theta (deg)')
plt.ylabel('I / (N^2 I0)')
plot_nicely()
plt.show()

<a id="fraunhofer-fft"></a>

$$
\text{用 FFT 计算夫琅禾费衍射: 孔径函数离散化, } x_n = n\Delta x \\
\hat{A}(f) = \int A(x) e^{-2\pi i f x} dx \approx \Delta x\sum_n A(x_n) e^{-2\pi i f x_n} \quad (\text{黎曼和}) \\
\text{在频率 } f_m = \frac{m}{N\Delta x} \text{ 上这个和就是离散傅里叶变换 (DFT), 用 FFT 快速计算} \\
\text{(np.fft.fftfreq 给出 } f_m \text{, fftshift 把零频移到中间; x 的原点不在第一个样本时多一个相位, 不影响模)} \\
\text{单缝的解析结果: } \hat{A}(f) = a\,\text{sinc}(af) = a\frac{\sin \pi af}{\pi af} \text{, 角度 } \sin\theta = \lambda f \\ \quad \\
\text{圆孔 (Circular Aperture, 半径 } a_r\text{, 直径 D): 二维傅里叶变换, 极坐标下积分得到贝塞尔函数 (Bessel Function)} \\
\hat{A}(\rho) = \int_0^{a_r}\int_0^{2\pi} e^{-2\pi i \rho r\cos\varphi} r \, d\varphi \, dr = 2\pi\int_0^{a_r} J_0(2\pi\rho r) \, r \, dr = \pi a_r^2 \cdot \frac{2 J_1(2\pi a_r\rho)}{2\pi a_r\rho} \\
\text{(用到 } J_0(x) = \frac{1}{2\pi}\int_0^{2\pi} e^{-ix\cos\varphi} d\varphi \text{ 和 } \frac{d}{dx}[x J_1(x)] = x J_0(x)\text{)} \\
I \propto \left[\frac{2 J_1(x)}{x}\right]^2, \ x = 2\pi a_r\rho = k a_r \sin\theta \quad (\text{艾里斑 Airy Disk}) \\ \quad \\
\text{第一个暗环: } J_1 \text{ 的第一个零点 } x = 3.8317 \implies \sin\theta = \frac{3.8317}{\pi} \cdot \frac{\lambda}{D} = 1.22\frac{\lambda}{D} \\
\text{瑞利判据 (Rayleigh Criterion): 两个点光源的艾里斑, 一个的中心落在另一个的第一暗环上时刚好能分辨} \\
\theta_{\min} \approx 1.22\frac{\lambda}{D} \quad (\text{望远镜, 显微镜, 相机的分辨率极限: 口径越大分辨率越高})
$$

数学基础: [数值积分 (黎曼和)](../math_basics/calculus.ipynb#integral) · [重积分与极坐标换元](../math_basics/calculus.ipynb#multiple-integral)

In [ ]:
# 1. 一维: 单缝的 FFT vs a sinc(a f)
n_samples, dx = 2**16, 0.05e-6
x = (np.arange(n_samples) - n_samples // 2) * dx
aperture = (np.abs(x) <= a / 2).astype(float)
A_hat = np.fft.fftshift(np.fft.fft(np.fft.ifftshift(aperture))) * dx    # ifftshift: 让 x = 0 对应第一个样本
f = np.fft.fftshift(np.fft.fftfreq(n_samples, d=dx))
analytic = a * np.sinc(a * f)
band = np.abs(f * wavelength) < 0.4           # sin(theta) = lambda f
print(f'单缝 FFT vs a sinc(a f): 最大误差 / a = {np.max(np.abs(np.abs(A_hat[band]) - np.abs(analytic[band]))) / a:.2e} (离散化误差 ~ dx / a = {dx / a:.0e})')

# 2. 二维: 圆孔的 FFT vs 艾里斑
N2, dx2 = 1024, 0.1e-6
r_ap = 3e-6
coords = (np.arange(N2) - N2 // 2) * dx2
X, Y = np.meshgrid(coords, coords)
disk = (X**2 + Y**2 <= r_ap**2).astype(float)
A2 = np.abs(np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(disk)))) * dx2**2
f2 = np.fft.fftshift(np.fft.fftfreq(N2, d=dx2))
row = A2[N2 // 2]                               # 沿 f_x 轴的截面
x_arg = 2 * np.pi * r_ap * f2
with np.errstate(invalid='ignore', divide='ignore'):
    airy = np.pi * r_ap**2 * np.where(x_arg == 0, 1.0, 2 * j1(x_arg) / x_arg)
sel = np.abs(f2) * wavelength < 0.4
print(f'圆孔 FFT vs 艾里公式: 最大相对误差 = {np.max(np.abs(row[sel] - np.abs(airy[sel]))) / (np.pi * r_ap**2):.2e}, 面积 (f = 0) = {row[N2 // 2]:.4e} vs pi r^2 = {np.pi * r_ap**2:.4e}')

first_zero = brentq(j1, 3, 4.5)
D = 2 * r_ap
print(f'J1 第一个零点 = {first_zero:.4f}, sin(theta) = {first_zero / np.pi:.4f} lambda / D = {first_zero / np.pi * wavelength / D:.5f}')

# 3. 瑞利判据: 两个非相干点光源, 强度相加
theta_min = 1.22 * wavelength / D
th = np.linspace(-3 * theta_min, 4 * theta_min, 2001)
airy_I = lambda t: (2 * j1(k * r_ap * np.sin(t) + 1e-12) / (k * r_ap * np.sin(t) + 1e-12))**2
total = airy_I(th) + airy_I(th - theta_min)
mid = airy_I(np.array([theta_min / 2]))[0] * 2
print(f'瑞利判据: 两个峰中间的强度 / 峰值 = {mid / total.max():.3f} (约 73.5%, 能看出两个峰)')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].imshow(np.log10(A2[N2 // 2 - 80:N2 // 2 + 80, N2 // 2 - 80:N2 // 2 + 80]**2 + 1e-30), cmap='inferno')
axs[0].set_title('log intensity: Airy pattern (2D FFT)')
axs[0].set_xticks([]); axs[0].set_yticks([])
for sep, name in [(0.6, 'unresolved'), (1.0, 'Rayleigh limit'), (1.6, 'resolved')]:
    axs[1].plot(th / theta_min, airy_I(th) + airy_I(th - sep * theta_min), label=f'{name} ({sep} theta_min)')
axs[1].set_xlabel('theta / theta_min')
axs[1].set_ylabel('total intensity')
plot_nicely()
plt.show()